## Estrutura mental do project 4 (Netflix)

```text
project4/
└── netflix
    ├── common.py        # funções comuns (probabilidades, log-likelihood, etc.)
    ├── em.py            # EM completo para mistura de Gaussianas com dados faltantes
    ├── naive_em.py      # EM “ingênuo” para dados completos (toy_data)
    ├── kmeans.py        # baseline com K-means
    ├── main.py          # script principal para rodar tudo e responder às questões
    ├── test.py          # script para testar sua implementação
    ├── toy_data.txt
    ├── netflix_incomplete.txt
    ├── netflix_complete.txt
    ├── test_incomplete.txt
    ├── test_complete.txt
    ├── test_solutions.txt
```

### O fluxo do projeto é mais ou menos:

1. Começar com toy_data.txt
	- Implementar K-means (kmeans.py)
	- Implementar naive_em.py (EM com dados completos)
	- Implementar funções em common.py (log-likelihood, etc.)

2. Depois ir para o problema Netflix
	- Implementar EM completo em em.py (lidando com entradas faltantes)
	- Usar main.py para rodar no netflix_incomplete.txt
	- Comparar com netflix_complete.txt
	- Validar com test_incomplete.txt, test_complete.txt, test_solutions.txt

## E-step

Para cada ponto $𝑥_𝑖$ e cada componente $𝑗$, calcular: $\quad post[i, j] = p(j | x_i) $

Usando Bayes:
$$ p(j | x_i) = \frac{P_j N(x_j; \mu_j, \sigma_j^2 I)}{\sum_{l=1}^k P_l N(x_j; \mu_l, \sigma_l^2 I)} $$

Onde:

- $P_j = $peso do componente
- $\mu_j =$ média
- $\sigma_j^2 = $ variância esférica
- $N(*) = $ densidade Gaussiana multivariada

### Densidade Gaussiana esférica

- **Para cada componente:**

$$ N(x_j; \mu_j, \sigma_j^2 I) = \frac{1}{(2\pi\sigma_j^2)^{d/2}}exp \bigg( -\frac{||x_i - \mu_j||^2}{2\sigma_j^2} \bigg) $$

### Likelihood

Para cada ponto: $\quad \log{\bigg(\sum_{j=1}^k \pi_j N(x_i; \mu_j, \sigma_j^2 I) \bigg)} $

Isso é exatamente: $ LL = \sum_{i-1}^n \log P(x_i | \theta) $

### Ou seja:

✔️ 1. A matriz ``post``
- Cada linha é uma distribuição sobre os clusters.

✔️ 2. A log-likelihood ``L``
- Usada para verificar convergência.

## M-step

Peso da mustira (proporções): 
- Se muitos pontos têm alta responsabilidade para o cluster $j$, então $P_j$ aumenta.
$$\quad P_j = \frac{1}{n} \sum_{i=1}^n p(j|x_i) $$

Média:
- A média é a média ponderada dos pontos, onde o peso é a responsabilidade.
$$ \quad \mu_j = \frac{\sum_{i=1}^n p(j|x_i)x_i}{\sum_{i=1}^n p(j|x_i)}  $$

Variâcias esféricas:
- A variância mede o espalhamento dos pontos em torno da nova média, também ponderado.
$$ \quad \sigma_j^2 = \frac{\sum_{i=1}^n p(j|x_i) \, ||x_i-\mu_j||^2}{d\sum_{i=1}^n p(j|x_i)} $$

## BIC Critério de informação bayesiano

$$ \text{BIC(M)} = l - \frac{1}{2}p \log n $$

- Aumentar a log‑verossimilhança $(LL)$  
    - → modelos com mais clusters explicam melhor os dados.
- Penalizar o número de parâmetros $(p)$  
    - → modelos com muitos clusters são mais complexos.

onde:
- $l$ = log‑verossimilhança final do EM
- $p$ = número de parâmetros livres
- $n$ = número de pontos de dados

## Implementando EM para preenchimento de matriz

Precisamos atualizar um pouco nosso algoritmo EM para lidar com o fato de que as observações não são mais  
vetores completos. Utilizamos a regra de Bayes para encontrar uma expressão atualizada para a probabilidade  
a posteriori $p(j|u) = P(y = j|x_{C_u}^{(u)}$:

$$ p(j\mid u) =\frac{p(u|j)\cdot p(j)}{p(u)} =\frac{p(u|j)\cdot p(j)}{\sum _{j=1}^{K}p(u|j)\cdot p(j)} =\frac{ \pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}}) }{ \sum _{j=1}^{K}\pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}}) } $$

Esta é a atribuição suave do cluster j para o ponto de dados $u$.

Para minimizar a instabilidade numerica, voce irá reimplementar a etapa E no domínio logarítmico, de modo  
que deve calcular os valores do logaritmo da probabilidade a posteriori, $  \ell (j, u) = \log (p (j|u))$ (embora a  
saída real da sua etapa E deva incluir a probabilidade a posteriori não logaritmada).

Seja $f(u,i) = log(\pi_i) + log \bigg(N (x_{C_u}^{(u)}; \mu_{C_u}^{(i)}, \sigma_i^2 I_{C_u \times C_u})\bigg)$. Então, em termos de f, a posteriori logarítmica é:

$$\displaystyle \ell (j|u)$$ $$  = \displaystyle \log (p(j\mid u)) = \log \left(\frac{ \pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}}) }{ \sum _{j=1}^{K}\pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}}) }\right)$$

$$ \displaystyle = \log \left(\pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}})\right) - \log \left(\sum _{j=1}^{K}\pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}})\right)$$

$$ \displaystyle =\log (\pi _{j})+\log \left(N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}})\right) - \log \left(\sum _{j=1}^{K}\exp (\log (\pi _{j}N(x_{C_{u}}^{(u)};\mu _{C_{u}}^{(j)},\sigma _{j}^{2}I_{C_{u}\times C_{u}})))\right)$$

$$ \displaystyle = f(u,j)-\log \left(\sum _{j=1}^{K}\exp (f(u,j))\right)$$




Uma vez que avaliamos $p (j|u)$ na etapa E, podemos prosseguir para a etapa M. Desejamos encontrar os  
parâmetros $\pi, \mu \text{ e } \sigma$, e o que maximizam $\ell (X; \theta)$,

Primeiro, note que, decompondo as Gaussianas esfericas multivariadas em Gaussianas esfericas univariadas  
como antes, podemos escrever, se $l \in C_u$:

$$\displaystyle  \frac{\partial }{\partial \mu _ l^{(k)}} N(x_{C_{u}}^{(u)}|\mu _{C_{u}}^{(k)},\sigma _{k}^{2}I_{|C_{u}|\times |C_{u}|})$$

$$\displaystyle  N(\dots ) \frac{ \frac{\partial }{\partial \mu _ l^{(k)}} \left(\frac{1}{\sqrt{2\pi } \sigma _{l,(k)}} \exp \Big(-\frac{1}{2\sigma _{l,(k)}^2} (x^{(u)}_ l - \mu _ l^{(k)})^2 \Big) \right) }{ \frac{1}{\sqrt{2\pi } \sigma _{l,(k)}} \exp \Big(-\frac{1}{2\sigma _{l,(k)}^2} (x^{(u)}_ l - \mu _ l^{(k)})^2 \Big) }$$

$$\displaystyle  N(\dots ) \frac{x_ l^{(u)} - \mu _ l^{(k)}}{\sigma _{l,(k)}^2}$$

onde $N(\dots ) = N(x_{C_{u}}^{(u)}|\mu _{C_{u}}^{(k)},\sigma _{k}^{2}I_{|C_{u}|\times |C_{u}|})$

Se $l \in C_u$, essa derivada é 0. Para abranger ambos os casos, podemos escrever:

$$\frac{\partial }{\partial \mu _ l^{(k)}} N(x_{C_{u}}^{(u)}|\mu _{C_{u}}^{(k)},\sigma _{k}^{2}I_{|C_{u}|\times |C_{u}|}) = N(x_{C_{u}}^{(u)}|\mu _{C_{u}}^{(k)},\sigma _{k}^{2}I_{|C_{u}|\times |C_{u}|}) \delta (l,C_ u) \frac{x_ l^{(u)} - \mu _ l^{(k)}}{\sigma _{l,(k)}^2}$$

onde $\delta(i, C_u)$ é uma função indicadora: 1 se $i \in C_u$ e zero caso contrário.



Seguindo a abordagem do algoritimo EM de maximizar uma função de verossimilhança aproximada $\hat{\ell }(X ; \theta )$  
durante a etapa M, considere a seguinte função:

$$\displaystyle  \hat{\ell }(X;\theta ) $$

$$\displaystyle =  \sum _{u = 1}^ n \sum _{j = 1}^ K p(j \mid u) \log \left(\frac{p\left( x^{(u)} \text { generated by cluster } j ; \theta \right)}{p(j \mid u)}\right)$$

$$\displaystyle  \sum _{u = 1}^ n \sum _{j = 1}^ K p(j \mid u) \log \left(\frac{\pi _ j \mathcal{N}(x_{C_ u}^{(u)} \mid \mu _{C_ u}^{(j)}, \sigma _ j^2 I_{|C_ u| \times |C_ u|})}{p(j \mid u)}\right),$$


onde $p (x^{(u)} \text{ generated by cluster } j; \theta)$ é a verossimilhança de $x^{(u)}$ gerado pelo cluster $j$ e o conjunto de  
parâmetros e $\theta$. Os valores $p (j| u)$ sao os que computamos na etapa E e sao constantes para a etapa M.

Agora, tomamos a derivada de $\hat{\ell} (X; \theta)$ em relação a $ \mu_l^{(k)} $ para encontrar o valor otimo de $\mu_l^{(k)}$ que maximiza $\hat{\ell} (X;0)$.

$$\displaystyle  - \frac{\partial }{\partial \mu _ l^{(k)}} \left[\sum _{u = 1}^ n \sum _{j = 1}^ K p(j \mid u) \cdot \frac{1}{2} \cdot \frac{\|  x_{C_ u}^{(u)} - \mu _{C_ u}^{(j)}\| ^2}{\sigma _ j^2} \right]$$

$$ \displaystyle  \sum _{u = 1}^ n p(k \mid u) \delta (l,C_ u) \frac{x_ l ^{(u)} - \mu _ l^{(k)}}{\sigma _ k^2}, $$


onde $\delta (i, C_u) = 1$ se $i \in C_u$ e $\delta (i, C_u) = 0 $ se $i \notin C_u$.


Igualando a derivada parcial a zero, obtemos que

$$\displaystyle  \widehat{\mu _ l^{(k)}} = \frac{\sum _{u = 1}^ n p(k \mid u) \delta (l,C_ u) x_ l^{(u)}}{\sum _{u=1}^ n p(k \mid u) \delta (l,C_ u)}.$$

Deixamos como exercicio ao leitor obter as estimativas de $\sigma_k^2$ e $\pi_k$ para $k = 1, ... , K$. Verifique que

$$\widehat{\sigma _ k^2} = \frac{1}{\sum _{u=1}^ n |C_ u| p(k \mid u)} \sum _{u = 1}^ n p(k \mid u) \| x_{C_ u}^{(u)} - \widehat{\mu _{C_ u}^{(k)}}\| ^2,$$

$$ \hat{\pi_k} = \frac{1}{n} \sum _{u=1}^ n p(k \mid u). $$

$$\sigma^2_{j,i} = \max(\sigma^2_{j,i}, \text{min_variance})$$